# Trace a field and analyze its path

Start with the generic tracer, which accepts analytic, model and interpolated
field callables. Then compare direction conventions with the geopack engine.
No model-specific tracing wrapper is copied into this notebook.

Run top to bottom in a fresh Python kernel after installing `.[examples]` from
    the repository root. No external data, network access, Qt or PyVista is required.
    Stored outputs are cleared; figures and assertions are generated by the cells.
    [Notebook index](README.md) · [Analysis guide](../../docs/geometry_analysis.md)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mageometry import trace_field_lines, geopack, geopack_field
from mageometry.geometry import field_line_curvature

def field(x, y, z):
    x, y, z = np.broadcast_arrays(x, y, z)
    return -y, x, np.ones_like(z)

## A helix with a known invariant

For B = (−y, x, 1), each line stays at fixed cylindrical radius. `direction=1`
follows B, `-1` follows −B, and `'both'` joins both halves around the seed.
`ds` is an integration step; `delta` below is a separate derivative step.
Unlike the geometry functions, tracer seed x/y/z arrays must have equal lengths.

In [ ]:
radii = np.array([.5, 1., 1.5])
trace = trace_field_lines(field, radii, np.zeros_like(radii), np.zeros_like(radii), direction='both', ds=.05,
                         err=1e-5, max_steps=80)
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
for i, radius in enumerate(radii):
    x, y, z = trace.path(i)
    s = trace.arc_length(i)
    np.testing.assert_allclose(np.hypot(x, y), radius, atol=2e-4)
    assert s[trace.start_index[i]] == 0 and np.all(np.diff(s) > 0)
    np.testing.assert_allclose(field_line_curvature(field, x, y, z, delta=.001),
                               radius/(1 + radius**2), atol=2e-4)
    ax.plot(x, y, z, label=f'r={radius:g}')
ax.set(xlabel='x', ylabel='y', zlabel='z')
ax.legend()
plt.show()

## Boundaries and termination

Use `bounds=grid.bounds` for a simulation grid and keep its default NaN fill.
Codes are 0: inner sphere, 1: outer sphere/box, 2: step limit,
3: undefined field, 4: custom stop. With `'both'`, `status` describes the +B
end and `status_backward` the −B end. Always inspect both ends.

In [ ]:
bounded = trace_field_lines(field, 1., 0., 0., direction='both', ds=.05,
                           bounds=((-2., 2.), (-2., 2.), (-.5, .5)), max_steps=100)
assert bounded.status[0] == 1 and bounded.status_backward[0] == 1
x, y, z = bounded.path(0)
np.testing.assert_allclose(z[[0, -1]], [-.5, .5], atol=1e-12)
print('Endpoints:', np.column_stack((x, y, z))[[0, -1]])

## The engine tracer has the opposite direction convention

`geopack.trace(..., dir=1)` integrates against B, while the generic tracer's
`direction=1` follows B. Engine paths and return values also differ. Retain
engine tracing for compatibility and scalar/vector regression work; use the
generic API for new geometry workflows.

The next small calculation checks the engine sign with a dot product, then
traces the same T89 + dipole model in the +B direction with the generic API. The algorithms
have different adaptive steps, so path nodes need not coincide.

In [ ]:
tilt = geopack.recalc(100.)
engine_field = geopack_field('t89', 'dip', parmod=4, ps=tilt)
seed = np.array([-5., 1., .5])
xe, ye, ze, xx, yy, zz = geopack.trace(*seed, 1., rlim=10., r0=2.,
                                     parmod=4, exname='t89', inname='dipole', maxloop=200)
path = np.column_stack((xx, yy, zz))
assert np.dot(path[1] - path[0], engine_field(*seed)) < 0
model_trace = trace_field_lines(engine_field, *seed, direction=1, ds=.05,
                                r0=2., rlim=10., max_steps=250)
print('Generic +B termination:', model_trace.status[0])

Full engine endpoint/path regressions live in `tests/test_trace_vectorized.py`
and `tests/test_trace_vectorized_with_vectorized_models.py`; generic tracer
regressions live in `tests/test_tracing.py`. See the
[geometry notebook](04_fieldline_geometry_and_derivatives.ipynb) for frame
validity and derivative convergence along a path.